<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/%D0%BA%D0%B8%D0%B1%D0%B5%D1%80%D0%BD%D0%B5%D1%82%D0%B8%D0%BA%D0%B03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

class ShannonFanoSequenceEncoder:
    def __init__(self, sequences, probabilities):
        self.sequences = sequences
        self.original_probabilities = probabilities.copy()
        self.codes = {}

    def build_codes(self, sequence_list, probabilities_dict, current_code=""):
        if len(sequence_list) == 1:
            self.codes[sequence_list[0]] = current_code
            return

        sorted_sequences = sorted(sequence_list,
                                key=lambda x: probabilities_dict[x],
                                reverse=True)

        total_prob = sum(probabilities_dict[s] for s in sorted_sequences)
        min_diff = float('inf')
        best_split_index = 1

        left_sum = 0
        for i in range(1, len(sorted_sequences)):
            left_sum += probabilities_dict[sorted_sequences[i-1]]
            right_sum = total_prob - left_sum
            diff = abs(left_sum - right_sum)
            if diff < min_diff:
                min_diff = diff
                best_split_index = i

        left_group = sorted_sequences[:best_split_index]
        right_group = sorted_sequences[best_split_index:]

        self.build_codes(left_group, probabilities_dict, current_code + "0")
        self.build_codes(right_group, probabilities_dict, current_code + "1")

    def encode_sequences(self):
        self.codes = {}
        self.build_codes(self.sequences, self.original_probabilities)

    def get_code_table(self):
        results = []
        for seq in sorted(self.sequences, key=lambda x: self.original_probabilities[x], reverse=True):
            prob = self.original_probabilities[seq]
            code = self.codes.get(seq, "")
            results.append({
                'Последовательность': seq,
                'Вероятность': f"{prob:.6f}",
                'Код': code
            })
        df = pd.DataFrame(results)
        return df

class Node:
    def __init__(self, sequence=None, probability=0):
        self.sequence = sequence
        self.probability = probability
        self.left = None
        self.right = None

def build_huffman_tree(nodes):
    nodes = nodes.copy()
    while len(nodes) > 1:
        nodes.sort(key=lambda x: x.probability)
        left = nodes.pop(0)
        right = nodes.pop(0)
        parent = Node(probability=left.probability + right.probability)
        parent.left = left
        parent.right = right
        nodes.append(parent)
    return nodes[0] if nodes else None

def assign_codes(node, current_code='', codes={}):
    if node is None:
        return codes
    if node.sequence is not None:
        codes[node.sequence] = current_code
        return codes
    assign_codes(node.left, current_code + '0', codes)
    assign_codes(node.right, current_code + '1', codes)
    return codes

def huffman_encode_sequences(sequences, probabilities):
    nodes = [Node(seq, prob) for seq, prob in zip(sequences, probabilities)]
    root = build_huffman_tree(nodes)
    codes = assign_codes(root, '', {})
    return codes

def main_analysis_task():
    text = """ Huffman coding
In computer science and information theory, a Huffman code is a particular type of optimal prefix code that is commonly used for
lossless data compression. The process of finding or using such a code is Huffman coding, an algorithm developed by David A. Huffman while he
was a Sc.D. student at MIT, and published in the 1952 paper "A Method for the Construction of Minimum-Redundancy Codes".[1]
The output from Huffman's algorithm can be viewed as a variable-length code table for encoding a source symbol (such as a character
in a file). The algorithm derives this table from the estimated probability or frequency of occurrence (weight) for each possible value of the source
symbol. As in other entropy encoding methods, more common symbols are generally represented using fewer bits than less common symbols.
Huffman's method can be efficiently implemented, finding a code in time linear to the number of input weights if these weights are sorted.[2]
However, although optimal among methods encoding symbols separately, Huffman coding is not always optimal among all compression methods
– it is replaced with arithmetic coding[3] or asymmetric numeral systems[4] if a better compression ratio is required.
The technique works by creating a binary tree of nodes. These can be stored in a regular array, the size of which depends on the number
of symbols.
A node can be either a leaf node or an internal node. Initially, all nodes are leaf nodes, which contain the symbol itself, the weight
(frequency of appearance) of the symbol and optionally, a link to a parent node which makes it easy to read the code (in reverse) starting from a
leaf node. Internal nodes contain a weight, links to two child nodes and an optional link to a parent node. As a common convention, bit '0' represents
following the left child and bit '1' represents following the right child. A finished tree has up to internal nodes. A Huffman tree that omits unused
symbols produces the most optimal code lengths.
The process begins with the leaf nodes containing the probabilities of the symbol they represent. Then, the process takes the two nodes
with smallest probability, and creates a new internal node having these two nodes as children. The weight of the new node is set to the sum of the
weight of the children. We then apply the process again, on the new internal node and on the remaining nodes (i.e., we exclude the two leaf nodes),
we repeat this process until only one node remains, which is the root of the Huffman tree.
The simplest construction algorithm uses a priority queue where the node with lowest probability is given highest priority:
Create a leaf node for each symbol and add it to the priority queue.
While there is more than one node in the queue:
Remove the two nodes of highest priority (lowest probability) from the queue
Create a new internal node with these two nodes as children and with probability equal to the sum of the two nodes' probabilities.
Add the new node to the queue.
The remaining node is the root node and the tree is complete.
Since efficient priority queue data structures require O(log n) time per insertion, and a tree with n leaves has 2n−1 nodes, this algorithm
operates in O(n log n) time, where n is the number of symbols.
If the symbols are sorted by probability, there is a linear-time (O(n)) method to create a Huffman tree using two queues, the first one
containing the initial weights (along with pointers to the associated leaves), and combined weights (along with pointers to the trees) being put in
the back of the second queue. This assures that the lowest weight is always kept at the front of one of the two queues:
Start with as many leaves as there are symbols.
Enqueue all leaf nodes into the first queue (by probability in increasing order so that the least likely item is in the head of the queue).
While there is more than one node in the queues:
Dequeue the two nodes with the lowest weight by examining the fronts of both queues.
Create a new internal node, with the two just-removed nodes as children (either node can be either child) and the sum of their weights
as the new weight.
Enqueue the new node into the rear of the second queue.
The remaining node is the root node; the tree has now been generated.
Once the Huffman tree has been generated, it is traversed to generate a dictionary which maps the symbols to binary codes as follows:
Start with current node set to the root.
If node is not a leaf node, label the edge to the left child as 0 and the edge to the right child as 1. Repeat the process at both the left child
and the right child.
The final encoding of any symbol is then read by a concatenation of the labels on the edges along the path from the root node to the
symbol.
In many cases, time complexity is not very important in the choice of algorithm here, since n here is the number of symbols in the
alphabet, which is typically a very small number (compared to the length of the message to be encoded); whereas complexity analysis concerns the
behavior when n grows to be very large.
It is generally beneficial to minimize the variance of codeword length. For example, a communication buffer receiving Huffman-encoded
data may need to be larger to deal with especially long symbols if the tree is especially unbalanced. To minimize variance, simply break ties between
queues by choosing the item in the first queue. This modification will retain the mathematical optimality of the Huffman coding while both
minimizing variance and minimizing the length of the longest character code.
Arithmetic coding and Huffman coding produce equivalent results — achieving entropy — when every symbol has a probability of the
form 1/2k. In other circumstances, arithmetic coding can offer better compression than Huffman coding because — intuitively — its "code words"
can have effectively non-integer bit lengths, whereas code words in prefix codes such as Huffman codes can only have an integer number of bits.
Therefore, a code word of length k only optimally matches a symbol of probability 1/2k and other probabilities are not represented optimally;
whereas the code word length in arithmetic coding can be made to exactly match the true probability of the symbol. This difference is especially
striking for small alphabet sizes.
"""

    target_sequences = ['th', 'tion', 'ing', 'ed', 're', 'are', 'is', 'an', 'the', 'to', 'and', 'it', 'of']

    occurrences = {}
    for seq in target_sequences:
        count = text.count(seq)
        occurrences[seq] = count

    total_occurrences = sum(occurrences.values())

    probabilities = {}
    for seq, count in occurrences.items():
        probabilities[seq] = count / total_occurrences if total_occurrences > 0 else 0.0

    # Удаление последовательностей с нулевым вхождением (если они есть)
    valid_sequences = [seq for seq in target_sequences if occurrences[seq] > 0]
    valid_probabilities = [probabilities[seq] for seq in valid_sequences]

    print(f"Общее число вхождений целевых последовательностей: {total_occurrences}")
    print(f"Количество уникальных последовательностей (с вхождениями): {len(valid_sequences)}\n")

    print("Таблица кодов (метод Шеннона-Фано):")
    print("+------------------+------------+--------+")
    print("|Последовательность| Вероятность| Код    |")
    print("+------------------+------------+--------+")

    shannon_fano = ShannonFanoSequenceEncoder(valid_sequences, dict(zip(valid_sequences, valid_probabilities)))
    shannon_fano.encode_sequences()
    fano_table = shannon_fano.get_code_table()

    for _, row in fano_table.iterrows():
        print(f"| {row['Последовательность']:<16} | {row['Вероятность']} | {row['Код']:<9} |")

    print("+------------------+------------+--------+\n")

    print("Таблица кодов (метод Хаффмана):")
    print("+------------------+------------+--------+")
    print("|Последовательность| Вероятность| Код    |")
    print("+------------------+------------+--------+")

    huffman_codes = huffman_encode_sequences(valid_sequences, valid_probabilities)
    sorted_sequences = sorted(valid_sequences, key=lambda x: probabilities[x], reverse=True)
    for seq in sorted_sequences:
        prob = probabilities[seq]
        code = huffman_codes[seq]
        print(f"| {seq:<16} | {prob:.6f} | {code:<9} |")

    print("+------------------+------------+--------+")

if __name__ == "__main__":
    main_analysis_task()

Общее число вхождений целевых последовательностей: 722
Количество уникальных последовательностей (с вхождениями): 13

Таблица кодов (метод Шеннона-Фано):
+------------------+------------+--------+
|Последовательность| Вероятность| Код    |
+------------------+------------+--------+
| th               | 0.239612 | 00        |
| the              | 0.164820 | 010       |
| re               | 0.126039 | 011       |
| it               | 0.090028 | 100       |
| an               | 0.087258 | 1010      |
| of               | 0.055402 | 1011      |
| ing              | 0.052632 | 1100      |
| is               | 0.049861 | 1101      |
| ed               | 0.041551 | 11100     |
| to               | 0.040166 | 11101     |
| and              | 0.024931 | 11110     |
| tion             | 0.015235 | 111110    |
| are              | 0.012465 | 111111    |
+------------------+------------+--------+

Таблица кодов (метод Хаффмана):
+------------------+------------+--------+
|Последовательность| Вероя